Example of using Orbit Fitter class and uncertainty ellipse visualization.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.compute as pc
from pathlib import Path

from adam_assist import ASSISTPropagator
from adam_core.orbit_determination import (
    OrbitDeterminationObservations,
    FittedOrbitMembers,
    FittedOrbits,
)
from adam_core.orbits.orbits import Orbits
from adam_core.orbits.plots import plot_orbit, add_observation_plot
from adam_core.orbit_determination import evaluate_orbits
from adam_fo.build import main as build_fo
from adam_fo.config import check_build_exists as check_fo_installed
from adam_fo.find_orb_orbit_fitter import FindOrbOrbitFitter
from adam_orbit_det_eval.utils import mpc_to_od_observations
from mpcq import MPCObservations
from mpcq.orbits import MPCOrbits

In [ ]:
# Load all the data. Sort observations by time so that residual plots are in chronological order
data_dir = Path("../../data/orbit_fit_eval")
mpc_observations = MPCObservations.from_parquet(
    data_dir / "mpc_observations.parquet"
).sort_by([("obstime", "ascending")])
mpc_orbits = MPCOrbits.from_parquet(data_dir / "mpc_orbits.parquet")
sbdb_orbits = Orbits.from_parquet(data_dir / "sbdb_orbits.parquet")
neocc_orbits = Orbits.from_parquet(data_dir / "neocc_orbits.parquet")

In [ ]:
# Evaluation helpers
def print_orbit_get_chi2(
    message: str,
    orbit: Orbits,
    observations: OrbitDeterminationObservations,
    propagator=ASSISTPropagator(),
    skip_ephem=False,
):
    if orbit is None or len(orbit) == 0:
        return None
    comm = orbit.coordinates.to_cometary()
    if not skip_ephem:
        fit_orbit, fit_members = evaluate_orbits(orbit, observations, propagator)
        rchi2, residuals = fit_orbit.reduced_chi2[0], fit_members.residuals
        chi2 = residuals.chi2.to_numpy()
    else:
        rchi2 = None
        chi2 = None
    np.set_printoptions(precision=3)
    print(
        f"{message}, object {orbit.object_id.to_pylist()}, orbit {orbit.orbit_id.to_pylist()},"
        f" rchi2={rchi2}, class {orbit.dynamical_class()}\n"
        f" q={comm.q.to_pylist()[0]} e={comm.e.to_pylist()[0]} i={comm.i.to_pylist()[0]}\n"
        f" raan={comm.raan.to_pylist()[0]} ap={comm.ap.to_pylist()[0]} tp={comm.tp.to_pylist()[0]}\n"
        f" time={comm.time.to_iso8601().to_pylist()[0]} ({comm.time.mjd()[0]} MJD)"
        f" period={comm.P[0]} origin={comm.origin.code.to_pylist()[0]}\n"
        f" sigmas in cometary {comm.covariance.sigmas[0]}"
    )
    return chi2


def show_for(
    object_id: str,
    od_observations: OrbitDeterminationObservations,
    fitted_orbits: FittedOrbits,
    fitted_members: FittedOrbitMembers,
    obsids,
) -> None:
    _, ax = plt.subplots(figsize=(15, 6))

    print(f"Object {object_id}, observation count {len(od_observations)}")
    res = print_orbit_get_chi2(
        "MPC orbit",
        mpc_orbits.apply_mask(
            pc.equal(mpc_orbits.requested_provid, object_id)
        ).orbits(),
        od_observations,
    )
    if res is not None:
        ax.plot(res, label="MPC")
    res = print_orbit_get_chi2(
        "SBDB orbit",
        sbdb_orbits.apply_mask(pc.match_substring(sbdb_orbits.object_id, object_id)),
        od_observations,
    )
    if res is not None:
        ax.plot(res, label="SBDB")
    res = print_orbit_get_chi2(
        "NEOCC orbit",
        neocc_orbits.apply_mask(
            pc.equal(neocc_orbits.object_id, object_id.replace(" ", ""))
        ),
        od_observations,
    )
    if res is not None:
        ax.plot(res, label="NEOCC")
    res = print_orbit_get_chi2(
        "Fitted orbit",
        fitted_orbits.apply_mask(pc.equal(fitted_orbits.object_id, object_id)),
        od_observations,
    )
    if res is not None:
        # Mark observations rejected by the fitter
        outliers = fitted_members.apply_mask(
            pc.is_in(fitted_members.obs_id, obsids)
        ).outlier
        ax.plot(res, markevery=outliers, marker="x", label="Fitted")
    ax.legend(loc="upper right")
    ax.set_title(f"Object {object_id} chi2 per observation")

    plt.show()

In [ ]:
try:
    check_fo_installed()
except Exception:
    build_fo()

fitter = FindOrbOrbitFitter(fo_result_dir="../../fo_dir")

In [ ]:
# Initial fit one object at a time
def initial_fit_one(object_id: str):
    subset = mpc_observations.apply_mask(
        pc.equal(mpc_observations.requested_provid, object_id)
    )
    od_observations = mpc_to_od_observations(subset, prevent_nans=False)
    fitted_orbits, fitted_members = fitter.initial_fit(object_id, od_observations)
    if len(fitted_orbits) > 0:
        # Recompute observations without NaNs to run evaluation
        od_observations = mpc_to_od_observations(subset, prevent_nans=True)
        show_for(
            object_id, od_observations, fitted_orbits, fitted_members, subset.obsid
        )
    else:
        print(f"No orbit fitted for object {object_id}")
    return fitted_orbits


initial_fitted_orbit = initial_fit_one(
    "2009 JY22"
)  # 81 observations, 35% with uncertainty

In [ ]:
# Plot position uncertainty for a few observations. Pick the blow up factor to make ellipses visible
def plot_observations(
    orbit: Orbits,
    observ: MPCObservations,
    radius_mult: float,
    propagator=ASSISTPropagator(),
):
    fig = plot_orbit(orbit, propagator)
    observed = propagator.propagate_orbits(
        orbit, observ.obstime, covariance=True
    ).coordinates
    add_observation_plot(fig, observed, radius_mult)
    fig.show()


object_id = "2009 JY22"
subset = mpc_observations.apply_mask(
    pc.equal(mpc_observations.requested_provid, object_id)
)
plot_observations(
    mpc_orbits.apply_mask(pc.equal(mpc_orbits.requested_provid, object_id)).orbits(),
    subset,
    5e4,
)
# plot_observations(sbdb_orbits.apply_mask(pc.match_substring(sbdb_orbits.object_id, object_id)), subset, 1e6)
# plot_observations(initial_fitted_orbit, subset, 1e6)